# Solutions — Databricks AI Functions (Medium 16)

> Requires Databricks Runtime with AI functions enabled.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_reviews    = spark.table("samples.bakehouse.media_customer_reviews")
df_franchises = spark.table("samples.bakehouse.sales_franchises")

## Solution 1 — Sentiment Analysis on Reviews (Sample 100)

In [ ]:
sample = (
    df_reviews
    .join(df_franchises.select("franchiseID", F.col("name").alias("franchise_name"), "country"),
          on="franchiseID")
    .limit(100)
)
result_1 = (
    sample
    .withColumn("sentiment", F.expr("ai_analyze_sentiment(review)"))
    .select("new_id","franchiseID","franchise_name","country","review","sentiment")
)

## Solution 2 — Sentiment Distribution per Franchise

In [ ]:
with_sentiment = (
    df_reviews
    .join(df_franchises.select("franchiseID", F.col("name").alias("franchise_name"), "country"),
          on="franchiseID")
    .withColumn("sentiment", F.expr("ai_analyze_sentiment(review)"))
)
total_per_franchise = with_sentiment.groupBy("franchiseID").agg(
    F.count("*").alias("total_reviews_franchise"))
result_2 = (
    with_sentiment
    .groupBy("franchiseID","franchise_name","country","sentiment")
    .agg(F.count("*").alias("numberReviews"))
    .join(total_per_franchise, on="franchiseID")
    .withColumn("percentageOfReviews",
        F.round(F.col("numberReviews") / F.col("total_reviews_franchise") * 100, 1))
    .drop("total_reviews_franchise")
    .orderBy("franchiseID","sentiment")
)

## Solution 3 — Top 5 Franchises by Positive Review Percentage

In [ ]:
with_sentiment = (
    df_reviews
    .join(df_franchises.select("franchiseID", F.col("name").alias("franchise_name"), "country"),
          on="franchiseID")
    .withColumn("sentiment", F.expr("ai_analyze_sentiment(review)"))
)
result_3 = (
    with_sentiment
    .groupBy("franchiseID","franchise_name","country")
    .agg(
        F.count("*").alias("total_reviews"),
        F.sum(F.when(F.col("sentiment") == "positive", 1).otherwise(0)).alias("positive_count")
    )
    .filter(F.col("total_reviews") >= 5)
    .withColumn("positive_pct", F.round(F.col("positive_count") / F.col("total_reviews") * 100, 1))
    .orderBy(F.col("positive_pct").desc())
    .limit(5)
)

## Solution 4 — Classify Review Topics

In [ ]:
# Why: ai_classify picks the best-fit label from the provided array
result_4 = (
    df_reviews
    .withColumn("topic",    F.expr("ai_classify(review, array('product quality','customer service','price','ambience','other'))"))
    .withColumn("sentiment",F.expr("ai_analyze_sentiment(review)"))
    .select("new_id","review","topic","sentiment")
)

## Solution 5 — Topic × Sentiment Heatmap

In [ ]:
with_labels = (
    df_reviews
    .withColumn("topic",    F.expr("ai_classify(review, array('product quality','customer service','price','ambience','other'))"))
    .withColumn("sentiment",F.expr("ai_analyze_sentiment(review)"))
)
result_5 = (
    with_labels
    .groupBy("topic","sentiment")
    .agg(F.count("*").alias("review_count"))
    .orderBy("topic","sentiment")
)